# CNN for Sentiment Analysis — NLTK Movie Reviews

A 1-D CNN slides filters over word embeddings to detect local phrases (e.g. "highly recommend", "waste of time") independent of position.

**Dataset**: NLTK `movie_reviews` — 2 000 labelled reviews (1 000 pos / 1 000 neg).

**Architecture**: Embedding → Conv1d (kernel sizes 3,4,5) → Global Max Pool → Dropout → Linear → Sigmoid

## Step 1: Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import nltk
import random
from collections import Counter

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print('device:', device)

## Step 2: Load the Movie Reviews Corpus

NLTK's `movie_reviews` has 2 000 documents already split by `pos` / `neg` categories. Each document is pre-tokenised into a list of words.

In [ ]:
from nltk.corpus import movie_reviews

random.seed(42)
docs = [(movie_reviews.words(fid), cat)
        for cat in movie_reviews.categories()
        for fid in movie_reviews.fileids(cat)]
random.shuffle(docs)

print(f'Total reviews : {len(docs)}')
print(f'Categories    : {movie_reviews.categories()}')
print(f'First 10 words: {list(docs[0][0])[:10]}')
print(f'Label         : {docs[0][1]}')

## Step 3: Build a Vocabulary

Cap at `MAX_VOCAB = 10 000` most frequent words. Reserve index 0 for `<PAD>` and index 1 for `<UNK>` (unknown words).

In [ ]:
MAX_VOCAB = 10000
MAX_LEN   = 200   # words per review

# TODO: count all lowercase words across all docs
all_words = ...
freq      = Counter(all_words)

# TODO: build vocab dict {'<PAD>':0, '<UNK>':1, word:idx, ...}
vocab = {'<PAD>': 0, '<UNK>': 1}
# add the top MAX_VOCAB-2 words

print(f'Vocab size: {len(vocab)}')
print(f'Top 10   : {freq.most_common(10)}')

## Step 4: Encode Reviews and Pad/Truncate

Map each word to its vocab index (1 = UNK if unseen). Pad with 0s or truncate so every review is exactly `MAX_LEN` words.

In [ ]:
def encode(words, vocab, max_len):
    # TODO: convert words to indices; truncate/pad to max_len
    tokens = ...
    return tokens

X_data = [encode(ws, vocab, MAX_LEN) for ws, _ in docs]
y_data = [1 if label == 'pos' else 0 for _, label in docs]

X_tensor = torch.tensor(X_data, dtype=torch.long)
y_tensor = torch.tensor(y_data, dtype=torch.float32)
print('X:', X_tensor.shape, '  y:', y_tensor.shape)

## Step 5: Train / Test Split and DataLoaders

In [ ]:
split = int(0.8 * len(X_tensor))
X_train, X_test = X_tensor[:split], X_tensor[split:]
y_train, y_test = y_tensor[:split], y_tensor[split:]

# TODO: wrap in TensorDataset and create DataLoaders
# batch_size=32, shuffle train set
train_loader = ...
test_loader  = ...
print(f'Train batches: {len(train_loader)}  Test batches: {len(test_loader)}')

## Step 6: Define the CNN Model

- `nn.Embedding(vocab_size, embed_dim)` — learnable word vectors
- `nn.Conv1d(embed_dim, num_filters, k)` — detect k-gram patterns (k=3,4,5)
- Global max pool across sequence — captures the strongest signal
- `nn.Dropout(0.5)` — regularisation
- `nn.Linear(num_filters * num_kernels, 1)` + sigmoid

⚠️ Embedding output is `(batch, seq, embed)` but Conv1d needs `(batch, embed, seq)` — remember to `.permute(0, 2, 1)`.

In [ ]:
class SentimentCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, num_filters=128, kernel_sizes=(3,4,5)):
        super().__init__()
        # TODO: embedding, convs (ModuleList), dropout, fc
        self.embedding = ...
        self.convs      = ...
        self.dropout    = ...
        self.fc         = ...

    def forward(self, x):
        x = self.embedding(x)       # (B, L, E)
        x = x.permute(0, 2, 1)     # (B, E, L)  for Conv1d
        # TODO: conv+relu, global max pool, concat, dropout, fc, sigmoid
        ...

vocab_size = len(vocab)
model = SentimentCNN(vocab_size).to(device)
print(model)

## Step 7: Train the Model

In [ ]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(10):
    model.train()
    total_loss = 0
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        # TODO: forward, loss, zero_grad, backward, step
        ...
    print(f'Epoch {epoch+1:2d} | Loss: {total_loss/len(train_loader):.4f}')

## Step 8: Evaluate on Test Set

In [ ]:
model.eval()
correct = total = 0
with torch.no_grad():
    for Xb, yb in test_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        # TODO: get predictions, count correct
        ...
print(f'Test accuracy: {correct/total*100:.1f}%')

## Step 9: Predict on a New Review

In [ ]:
def predict(text, model, vocab, max_len, device):
    model.eval()
    with torch.no_grad():
        tokens = [vocab.get(w.lower(), 1) for w in text.split()][:max_len]
        tokens += [0] * (max_len - len(tokens))
        x = torch.tensor([tokens], dtype=torch.long, device=device)
        prob = model(x).item()
        return f'positive ({prob:.2f})' if prob >= 0.5 else f'negative ({prob:.2f})'

print(predict('This film was absolutely wonderful and moving', model, vocab, MAX_LEN, device))
print(predict('Terrible waste of time, boring and predictable', model, vocab, MAX_LEN, device))